In [1]:
import torch 
import torch.nn.functional as F 
import matplotlib.pyplot as plt 
%matplotlib inline 


In [2]:
words = open("names (1).txt" ,"r").read().splitlines()
words[:8]


['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
chars= sorted(list(set(''.join(words))))
mapping  = {c : i+1  for i,c in enumerate(chars)}
mapping['.'] = 0
reverse_mapping = {i:c for c,i in mapping.items()}
reverse_mapping


{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [5]:
block_size = 3
X , y = [] , []
for w in words[:5]:
    print(w)
    context = [0]*block_size
    for ch in w + '.':
        ix = mapping[ch]
        X.append(context)
        y.append(ix)
        print(''.join(reverse_mapping[i] for i in context) ,'---->' , reverse_mapping[ix])
        context = context[1:] + [ix]
        
X=torch.tensor(X)
y=torch.tensor(y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [6]:
X.shape , y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [7]:
C=torch.randn((27,2))


In [8]:
F.one_hot(torch.tensor(5) , num_classes= 27).float() @ C

tensor([ 0.6451, -1.5153])

In [16]:
#embedding X for 27 by 2 indexing
emb = C[X]


In [31]:
w1 = torch.rand((6 ,100))
b = torch.rand(100)

In [27]:
torch.cat([emb[: , 0 , :] , emb[: , 1 , :] , emb[: , 2 , :] ] , 1).shape

torch.Size([32, 6])

In [28]:
#this work when we change the block size 
torch.cat(torch.unbind(emb , 1) , 1).shape

torch.Size([32, 6])

In [38]:
h = torch.tanh(emb.view(-1 , 6) @ w1 + b)
h


tensor([[ 0.9689,  0.9978,  0.9988,  ...,  0.9321,  0.9967,  0.6346],
        [-0.2921,  0.9552,  0.8341,  ...,  0.8566,  0.9916,  0.1828],
        [-0.2147, -0.2171, -0.1431,  ...,  0.5608,  0.7012,  0.4319],
        ...,
        [-0.2691,  0.7212, -0.7272,  ..., -0.3537,  0.2434,  0.1804],
        [-0.9555, -0.9714, -0.9884,  ..., -0.5954, -0.8022, -0.8694],
        [-0.4373, -0.8518, -0.8724,  ..., -0.8159, -0.9396, -0.8776]])

In [39]:
#creating the final layer or output layer
W2 = torch.randn((100 , 27))
b2 = torch.randn(27)

In [40]:
logits = h @ W2  + b2

In [42]:
logits

tensor([[ 6.4133e+00, -4.8258e-01, -2.1537e+00, -1.2874e+00,  3.4786e+00,
          1.0430e+01, -8.8735e+00, -6.1899e+00,  8.4294e-01, -2.1233e+00,
          8.4531e+00,  4.1819e+00, -6.9033e-01,  4.4095e-01, -2.7389e+00,
          1.5548e+01,  8.9585e+00,  2.4763e+00, -1.7731e+01,  4.5943e+00,
         -2.7929e+00, -3.9631e+00,  8.2644e+00, -2.1432e+01, -4.6361e+00,
         -3.3750e+00,  6.5814e+00],
        [ 2.8616e+00, -1.4365e+00, -3.6345e+00, -2.8805e+00,  3.8274e+00,
          1.2662e+01, -9.8003e+00, -7.6565e+00, -1.8379e+00,  1.5473e+00,
          2.9879e+00, -6.2745e-01, -1.9779e-01,  2.7505e+00, -7.2644e-01,
          8.8460e+00,  7.8784e+00,  8.4103e-01, -1.5595e+01,  5.2773e+00,
         -2.2953e+00, -3.8067e+00,  2.3487e+00, -1.9425e+01, -8.9737e+00,
         -2.9001e+00,  6.7036e+00],
        [ 4.1096e+00, -1.2168e+01,  1.0458e+00,  4.7669e+00,  1.1285e+00,
          4.2464e+00,  1.5381e-01, -7.1969e+00, -2.7492e+00,  2.9871e+00,
         -1.7708e+00, -8.7257e-01,  1.73

In [43]:
counts = logits.exp()

In [46]:
prob = counts / counts.sum(1 , keepdim=True)
prob[0].sum()

tensor(1.)

In [ ]:
prob[torch.arange(32) , y ]

tensor([5.9373e-03, 4.7967e-05, 1.0892e-09, 3.9993e-07, 1.1564e-02, 9.9093e-01,
        2.7501e-07, 1.1028e-04, 1.6261e-08, 1.1495e-09, 1.4724e-03, 3.3640e-01,
        1.0821e-07, 6.0207e-04, 4.8215e-08, 2.5219e-04, 2.0975e-08, 3.2302e-02,
        1.3109e-10, 6.4492e-06, 5.3036e-03, 3.0922e-06, 1.3503e-06, 1.2082e-07,
        3.3993e-03, 1.7343e-05, 9.9369e-01, 8.6277e-03, 3.5801e-06, 5.9637e-02,
        6.4994e-07, 1.7151e-05])

In [48]:
loss =  - prob[torch.arange(32) , y ].log().mean()
loss

tensor(10.5944)